# tp-08 馬券最適化の強化学習（予測結果 → 賭け方の方策）

**実行場所: ホスト PC（stg）。** tp-05/06 の予測（勝率・連対率・複勝率）と、過去のオッズ・払戻の表（`data/local/modeling/betting/`）から、レースごとの賭け金の配分を学習する。

## 問題の設定
- 1 レース = 1 ステップの文脈つきバンディット。**状態** = そのレースの候補の買い目（券種 × 組。単勝・複勝・馬連・ワイド・馬単）と特徴量、**行動** = 各買い目への賭け金の割合（予算 1 = UI 既定の 1R 2,000 円。合計 ≤ 1、残りは見送り）、**報酬** = 払戻 − 賭け金。
- 買い目の確率は、モデルの勝率・連対率・複勝率に整合する割引 Harville（`exotic_probabilities`）。**市場の暗黙の確率とのブレンド**（Benter 型 `p ∝ p_model^a · p_market^b`、a・b は 2022 年だけで当てはめる）も使う。
- 全買い目の結果が事後に分かる（full information）ので、方策勾配を期待値で厳密に計算する **expected** と、行動をサンプリングする **REINFORCE**（基準線 = 厳密な期待値）の両方を実装して比べる（try-both）。
- 基準（ベースライン）: 単勝 1 点（モデル 1 位／市場 1 番人気）、EV しきい値、EV 上位 k 点、EV 上位レースだけ（参加率をそろえる）、1/4 Kelly、ランダム。
- **見送りを自由にすると、期待値が 1 未満なら「全部見送る」が最適になる**（実際にそうなった。§5）。推奨馬券を出す用途では、**1 レース当たりの平均賭け金の下限**（`target_stake`、予算比。0.5 = 1R 平均 1,000 円）を罰則で課し、
  その条件のもとで損失を最小（回収率を最大）にする方策を学習する。下限 0（自由）・0.25・0.5・1.0（毎レース全額）の 4 水準を比べる。

## 評価の分け方（test 2025 は一度だけ）
- 方策の学習 2022 → 評価 2023（fold A）、学習 2022〜2023 → 評価 2024（fold B）。ハイパーパラメータ（Optuna）は A・B の平均損益で選ぶ。
- 最終: 学習 2022〜2024（5 seed の配分を平均）→ **test 2025 を一度だけ評価**（印 `rl/TEST2025_EVALUATED_<tag>.json`）。

## 注意（結果の読み方）
- **オッズは締切後の確定オッズ**（2020〜2025 年のスナップショットは無い）。購入判断の時点（T-60/T-15）の値とは違うため、回収率は楽観側に偏りうる。
- 三連複・三連単はオッズが無いため対象外。複勝・ワイドは判断に下限オッズを使い、払戻は確定値。
- 控除率は単勝・複勝 20%、馬連・ワイド 22.5%、馬単 25%。市場が効率的なら、何も工夫しない賭け方の回収率は 75〜80% になる。

In [ ]:
import os, sys
from pathlib import Path


def _find_root():
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / "AGENTS.md").exists() and (q / "src").is_dir():
            return q
    raise RuntimeError("リポジトリルートが見つかりません")


REPO_ROOT = _find_root()
os.chdir(REPO_ROOT)
os.environ.setdefault("POLARS_MAX_THREADS", "6")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from src.utils.project_env import load_project_dotenv
load_project_dotenv()

In [ ]:
import json, subprocess, time, gc
from datetime import datetime
import numpy as np
import pandas as pd
import mlflow
import optuna

from src.pipeline.betting import policy as pol
from src.pipeline.betting import ticket_env as te
from src.pipeline.mlflow.train_utils import require_tracking_server
from src.pipeline.models.compute import detect_compute

WORK = REPO_ROOT / "data/local/modeling/training_pipeline"
BET = REPO_ROOT / "data/local/modeling/betting"
TAG = os.environ.get("TP08_TAG", "ens_v1")
OOF_FILE = Path(os.environ.get("TP08_OOF", WORK / "oof/model_oof/ensemble.parquet"))
TEST_FILE = Path(os.environ.get("TP08_TEST", WORK / "final_eval/test2025_predictions.parquet"))
K = 12
OUT = BET / "rl" / TAG
OUT.mkdir(parents=True, exist_ok=True)
LOCK = BET / "rl" / f"TEST2025_EVALUATED_{TAG}.json"
THREADS = max(1, min(4, detect_compute(reserve_cores=6).threads))
GIT_COMMIT = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True, cwd=REPO_ROOT).stdout.strip()
mlflow.set_experiment("keiba-tp-betting-rl")
require_tracking_server()
YEARS = ["2022", "2023", "2024", "2025"]
wp = pd.read_parquet(BET / "odds_win_place.parquet")
wp = wp[wp["race_id"].str[:4].isin(YEARS)].reset_index(drop=True)
pair = pd.read_parquet(BET / "odds_pair.parquet")
pair = pair[pair["race_id"].str[:4].isin(YEARS)].reset_index(drop=True)
pay = pd.read_parquet(BET / "payoff.parquet")
pay = pay[pay["race_id"].str[:4].isin(YEARS)].reset_index(drop=True)
cols = ["race_id", "horse_id", "year", "p_win", "p_top2", "p_top3"]
pred_all = pd.concat([pd.read_parquet(OOF_FILE)[cols], pd.read_parquet(TEST_FILE)[cols]], ignore_index=True)
pred_all = pred_all[pred_all["year"].between(2022, 2025)]
print(f"予測 {len(pred_all):,} 行 / オッズ {len(wp):,} 行 / 組オッズ {len(pair):,} 行 / 払戻 {len(pay):,} 行 / スレッド {THREADS}")
print("予測元:", OOF_FILE.name, TEST_FILE.name, "| タグ:", TAG)

## 1. 市場の暗黙の確率とのブレンド（Benter 型）
勝ち馬の対数尤度が最大になる (a, b) を **2022 年だけ**で当てはめ、2023・2024 年で「モデル単独」「市場単独」「ブレンド」を比べる。

In [ ]:
win_no = pay[pay["bet_type"] == "tansho"].groupby("race_id")["combo"].agg(["count", "first"])
win_no = win_no[win_no["count"] == 1]["first"].astype(int)
m = pred_all.merge(wp[["race_id", "horse_id", "horse_number", "win_odds"]], on=["race_id", "horse_id"], how="inner").dropna(subset=["win_odds"])
m = m[m["race_id"].isin(win_no.index)]
m["won"] = m["horse_number"].to_numpy() == win_no.reindex(m["race_id"]).to_numpy()
groups = {}
for rid, g in m.groupby("race_id", sort=False):
    if g["won"].sum() != 1:
        continue
    pm = g["p_win"].to_numpy(); pm = pm / pm.sum()
    pk = te.market_probs(g["win_odds"].to_numpy())
    groups[rid] = (np.log(np.clip(pm, 1e-12, None)), np.log(pk), int(np.flatnonzero(g["won"].to_numpy())[0]))
by_year = {y: [v for r, v in groups.items() if r[:4] == y] for y in YEARS}
A_B, B_B = te.fit_benter(by_year["2022"])
print(f"Benter (a, b) = ({A_B:.3f}, {B_B:.3f})  ← 2022 年 {len(by_year['2022']):,} レースで当てはめ")
def nll(gs, a, b):
    s = 0.0
    for lm, lk, w in gs:
        z = a * lm + b * lk
        mx = z.max(); s -= z[w] - mx - np.log(np.exp(z - mx).sum())
    return s / len(gs)
rows = []
for y in ["2023", "2024", "2025"]:
    gs = by_year[y]
    rows.append({"year": y, "races": len(gs), "モデル単独": nll(gs, 1, 0), "市場単独": nll(gs, 0, 1), "ブレンド": nll(gs, A_B, B_B)})
bl = pd.DataFrame(rows).set_index("year")
display(bl.round(4))
bl.to_csv(OUT / "benter_blend.csv")

## 2. 候補の買い目の表を作る
各レース・各券種で、確率の上位 K=12 点と期待回収率（p×オッズ）の上位 6 点の和集合。年ごとに保存（タグ・K・(a, b) が同じなら再利用）。

In [ ]:
TDIR = BET / "tickets" / f"{TAG}_K{K}"
TDIR.mkdir(parents=True, exist_ok=True)
TICK = {}
for y in YEARS:
    f = TDIR / f"{y}.parquet"
    meta_f = TDIR / f"{y}.json"
    want = {"a": round(A_B, 6), "b": round(B_B, 6), "K": K}
    if f.exists() and meta_f.exists() and json.loads(meta_f.read_text()) == want:
        TICK[y] = pd.read_parquet(f)
        print(y, "キャッシュ", len(TICK[y]))
        continue
    t0 = time.time()
    ids = set(pred_all.loc[pred_all["year"] == int(y), "race_id"])
    d = te.build_parallel(pred_all[pred_all["race_id"].isin(ids)], wp[wp["race_id"].isin(ids)], pair[pair["race_id"].isin(ids)], pay[pay["race_id"].isin(ids)], A_B, B_B, K=K, procs=THREADS, chunks=40)
    d.to_parquet(f, index=False); meta_f.write_text(json.dumps(want))
    TICK[y] = d
    print(f"{y}: 予測のレース {len(ids):,} → 買い目の表のレース {d['race_id'].nunique():,}、{len(d):,} 点 ({time.time() - t0:.0f}s)")
cov = pd.DataFrame({y: {"races": d["race_id"].nunique(), "tickets": len(d), "tickets/race": len(d) / d["race_id"].nunique()} for y, d in TICK.items()}).T
display(cov.round(1)); cov.to_csv(OUT / "ticket_coverage.csv")

## 3. 市場に対する優位はあるか（期待回収率の較正）
買い目を期待回収率（モデルのブレンド確率 × オッズ）の帯に分け、実際の回収率（的中時の払戻倍率の平均）を見る。優位があるなら、期待回収率が高い帯ほど実際の回収率も高く、1 を超える帯がある。
モデル単独の確率（`p_model`）での同じ表も出す（ブレンドの効果を見る）。

In [ ]:
VAL = pd.concat([TICK["2023"], TICK["2024"]], ignore_index=True)
bins = [-1, 0.6, 0.8, 0.9, 1.0, 1.1, 1.25, 1.5, 2.0, 1e9]
def calib(col):
    d = VAL.assign(x=VAL[col] * VAL["odds"])
    d["band"] = pd.cut(d["x"], bins)
    return d.groupby(["type", "band"], observed=True).agg(n=("mult", "size"), 期待回収率=("x", "mean"), 実際の回収率=("mult", "mean")).round(3)
cal_blend, cal_model = calib("p_comb"), calib("p_model")
display(cal_blend.unstack(0)["実際の回収率"].round(3))
print("↑ ブレンド確率での期待回収率の帯 × 券種の、実際の回収率（valid 2023・2024）")
display(cal_model.unstack(0)["実際の回収率"].round(3))
print("↑ モデル単独の確率での同じ表")
cal_blend.to_csv(OUT / "ev_calibration_blend.csv"); cal_model.to_csv(OUT / "ev_calibration_model.csv")

## 4. 基準となる賭け方（valid 2023・2024）
EV しきい値 τ は学習側の年（2023 の基準は 2022、2024 の基準は 2022〜2023）で選び、評価年に当てはめる。

In [ ]:
def packed(years, mu=None, sd=None):
    d = pd.concat([TICK[y] for y in years], ignore_index=True)
    return pol.pack(d, mu, sd)
TAUS = [0.0, 0.05, 0.1, 0.2, 0.3, 0.5]
def choose_tau(train_years):
    pk, _, _ = packed(train_years)
    best = max(TAUS, key=lambda t: pol.summarize(pol.settle(pk, pol.alloc_ev_threshold(pk, t)), n_boot=1)["profit_per_race"])
    return best
def baselines(pk, tau):
    return {
        "単勝: モデル1位に全額": pol.alloc_top_p(pk, "tansho"),
        "単勝: 1番人気に全額": pol.alloc_favorite(pk, "tansho"),
        "複勝: モデル1位に全額": pol.alloc_top_p(pk, "fukusho"),
        "馬連: モデル1位に全額": pol.alloc_top_p(pk, "umaren"),
        f"EV≥{tau:.2f} に等分": pol.alloc_ev_threshold(pk, tau),
        f"EV≥{tau:.2f} の上位3点に等分": pol.alloc_ev_threshold(pk, tau, top=3),
        "EV上位25%のレースだけ・上位3点": pol.alloc_ev_selective(pk, 0.25, 3),
        "EV上位50%のレースだけ・上位3点": pol.alloc_ev_selective(pk, 0.5, 3),
        "1/4 Kelly": pol.alloc_kelly(pk, 0.25),
        "ランダム1点": pol.alloc_random(pk),
    }
FOLDS = [("A", ["2022"], "2023"), ("B", ["2022", "2023"], "2024")]
base_rows = []
for fold, tr_y, va_y in FOLDS:
    tau = choose_tau(tr_y)
    pk, _, _ = packed([va_y])
    for name, w in baselines(pk, tau).items():
        s = pol.summarize(pol.settle(pk, w), n_boot=500)
        base_rows.append({"fold": f"{fold} (valid {va_y})", "方策": name, **{k: s[k] for k in ("roi", "roi_lo", "roi_hi", "profit_per_race", "participation")}})
    print(f"fold {fold}: EV しきい値 τ = {tau}")
base_df = pd.DataFrame(base_rows)
display(base_df.pivot(index="方策", columns="fold", values="roi").round(3))
base_df.to_csv(OUT / "baselines_valid.csv", index=False)

## 5. 方策の学習: expected と REINFORCE の比較（既定のハイパーパラメータ、2 seed）

In [ ]:
def run_fold(cfg, tr_years, va_year, seeds=(0, 1)):
    pk_tr, mu, sd = packed(tr_years)
    pk_va, _, _ = packed([va_year], mu, sd)
    ws = []
    for sd_ in seeds:
        c = pol.TrainConfig(**{**cfg.__dict__, "seed": sd_, "threads": THREADS})
        net, hist = pol.train_policy(pk_tr, c)
        ws.append(pol.predict_alloc(net, pk_va, c.max_ticket))
    w = np.mean(ws, axis=0)
    return pol.settle(pk_va, w), pol.summarize(pol.settle(pk_va, w), n_boot=500), pk_va, w
cmp_rows = []
for mode in ("expected", "reinforce"):
    for tgt in (0.0, 0.5):
        for fold, tr_y, va_y in FOLDS:
            t0 = time.time()
            _, s, _, _ = run_fold(pol.TrainConfig(mode=mode, target_stake=tgt), tr_y, va_y)
            cmp_rows.append({"mode": mode, "賭け金の下限": tgt, "fold": fold, **{k: s[k] for k in ("roi", "roi_lo", "roi_hi", "profit_per_race", "stake_per_race", "participation")}, "sec": time.time() - t0})
cmp_df = pd.DataFrame(cmp_rows)
display(cmp_df.round(4)); cmp_df.to_csv(OUT / "mode_comparison.csv", index=False)
print("↑ 下限 0（自由）では賭け金がほぼ 0（全部見送り）に落ちる。下限を課すと、その水準の賭け金のもとで回収率を最大化する。")

## 6. ハイパーパラメータの最適化（Optuna）
賭け金の下限 0.5（1R 平均 1,000 円）を課したもとで、目的 = fold A・B の損益（予算 1 に対する 1 レース当たり）の平均。下限（の 9 割）を割った試行には罰則。モード（expected / reinforce）も探索に含める。

In [ ]:
N_TRIALS = int(os.environ.get("TP08_TRIALS", "30"))
TARGET_OPT = 0.5
def make_cfg(t):
    return pol.TrainConfig(mode=t.suggest_categorical("mode", ["expected", "reinforce"]), hidden=t.suggest_categorical("hidden", [32, 64, 128]),
                           dropout=t.suggest_float("dropout", 0.0, 0.3), lr=t.suggest_float("lr", 1e-3, 1e-2, log=True), wd=t.suggest_float("wd", 1e-5, 1e-2, log=True),
                           epochs=t.suggest_int("epochs", 10, 60), batch_races=t.suggest_categorical("batch_races", [128, 256, 512]),
                           entropy=t.suggest_float("entropy", 1e-4, 0.1, log=True), risk=t.suggest_float("risk", 0.0, 2.0),
                           max_ticket=t.suggest_categorical("max_ticket", [0.3, 0.5, 1.0]), target_stake=TARGET_OPT)
def objective(t):
    cfg = make_cfg(t)
    vals = []
    for fold, tr_y, va_y in FOLDS:
        _, s, _, _ = run_fold(cfg, tr_y, va_y, seeds=(0, 1))
        vals.append(s["profit_per_race"] - 5.0 * max(0.0, 0.9 * TARGET_OPT - s["stake_per_race"]))
        t.set_user_attr(f"roi_{fold}", s["roi"]); t.set_user_attr(f"part_{fold}", s["participation"])
    return float(np.mean(vals))
optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=0))
t0 = time.time()
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)
trials = study.trials_dataframe()
display(trials.sort_values("value", ascending=False).head(8)[["number", "value", "params_mode", "params_hidden", "params_epochs", "params_entropy", "params_risk", "params_max_ticket", "user_attrs_roi_A", "user_attrs_roi_B", "user_attrs_part_A", "user_attrs_part_B"]].round(4))
BEST = {**study.best_params, "target_stake": TARGET_OPT}
print(f"最良 {study.best_value:.5f}（{time.time() - t0:.0f}s）:", BEST)
trials.to_csv(OUT / "optuna_trials.csv", index=False)
(OUT / "best_config.json").write_text(json.dumps({"params": BEST, "value": study.best_value, "n_trials": N_TRIALS}, ensure_ascii=False, indent=1))

In [ ]:
cfg_best = pol.TrainConfig(**BEST)
rows, folds_res = [], {}
for fold, tr_y, va_y in FOLDS:
    s_df, s, pk_va, w = run_fold(cfg_best, tr_y, va_y, seeds=(0, 1, 2, 3, 4))
    folds_res[fold] = (s_df, pk_va, w)
    rows.append({"fold": f"{fold} (valid {va_y})", **{k: s[k] for k in ("roi", "roi_lo", "roi_hi", "profit_per_race", "stake_per_race", "participation", "max_drawdown")}})
best_df = pd.DataFrame(rows).set_index("fold")
display(best_df.round(4)); best_df.to_csv(OUT / "best_valid.csv")
print("↑ 最良設定の 5 seed 平均の配分（valid）。ハイパーパラメータは同じ valid で選んだので、やや楽観的。")

## 7. 最終: 学習 2022〜2024 → test 2025（一度だけ）
賭け金の下限 0・0.25・0.5・1.0 のそれぞれについて、最良設定で 5 seed の配分の平均を使う。ベースラインの τ は 2022〜2024 で選ぶ。ここで初めて 2025 年の払戻を使う。

In [ ]:
SMOKE = os.environ.get("TP08_SMOKE") == "1"          # 動作確認用: test 2025 を使わず、2022〜2023 で学習・2024 で評価する
if not SMOKE and LOCK.exists() and os.environ.get("TP08_FORCE") != "1":
    raise SystemExit(f"test 2025 は評価済みです（{LOCK}）。結果を見てやり直さないため、再実行しません。")
TEST_Y = "2024" if SMOKE else "2025"
TARGETS = [0.0, 0.25, 0.5, 1.0]
HEAD = "RL（下限0.5）"
TR_Y = ["2022", "2023"] if SMOKE else ["2022", "2023", "2024"]
pk_tr, mu, sd = packed(TR_Y)
pk_te, _, _ = packed([TEST_Y], mu, sd)
W = {}
for tgt in TARGETS:
    ws = []
    for seed in range(5):
        c = pol.TrainConfig(**{**cfg_best.__dict__, "target_stake": tgt, "seed": seed, "threads": THREADS})
        net, hist = pol.train_policy(pk_tr, c)
        ws.append(pol.predict_alloc(net, pk_te, c.max_ticket))
    W[tgt] = np.mean(ws, axis=0)
tau = choose_tau(TR_Y)
ALLOCS = {f"RL（下限{t}）": W[t] for t in TARGETS} | baselines(pk_te, tau)
SETTLED = {k: pol.settle(pk_te, w) for k, w in ALLOCS.items()}
fin_rows = []
for k, s in SETTLED.items():
    r = pol.summarize(s, n_boot=2000)
    fin_rows.append({"方策": k, **{c: r[c] for c in ("roi", "roi_lo", "roi_hi", "profit_per_race", "stake_per_race", "participation", "max_drawdown")}})
fin_df = pd.DataFrame(fin_rows).set_index("方策")
display(fin_df.round(4)); fin_df.to_csv(OUT / "final_test2025.csv")
print(f"EV しきい値 τ = {tau}（2022〜2024 で選択） / テストのレース数 {pk_te.n_races:,}")

In [ ]:
# 回収率の差（RL − 相手）: レース単位の paired bootstrap（賭け金が違う方策どうしなので、損益ではなく回収率 = 総払戻 / 総賭け金の差）
rng = np.random.default_rng(0)
RID = pk_te.race_ids
def arrs(k):
    s = SETTLED[k].set_index("race_id").reindex(RID)
    return s["ret"].to_numpy(), s["stake"].to_numpy()
def roi_diff(ka, kb, n_boot=2000):
    ra, sa = arrs(ka); rb, sb = arrs(kb)
    f = lambda ii: ra[ii].sum() / max(sa[ii].sum(), 1e-12) - rb[ii].sum() / max(sb[ii].sum(), 1e-12)
    d = f(np.arange(len(RID)))
    bs = np.array([f(rng.integers(0, len(RID), len(RID))) for _ in range(n_boot)])
    return d, np.quantile(bs, 0.025), np.quantile(bs, 0.975)
rows = []
for k in SETTLED:
    if k.startswith("RL"):
        continue
    row = {"比較相手": k}
    for t in (0.5, 1.0):
        d, lo, hi = roi_diff(f"RL（下限{t}）", k)
        row[f"下限{t}: 回収率の差"], row[f"下限{t}: 95%下限"], row[f"下限{t}: 95%上限"] = d, lo, hi
    rows.append(row)
diff_df = pd.DataFrame(rows).set_index("比較相手")
display(diff_df.round(4)); diff_df.to_csv(OUT / "final_vs_baselines.csv")
# 券種別（各下限の賭け金の配分と回収率）
tt_rows = []
for t in TARGETS:
    w_ = np.where(pk_te.mask, W[t], 0.0)
    for ti, name in enumerate(te.TYPES):
        m = pk_te.ttype == ti
        stake_, ret_ = float((w_ * m).sum()), float((w_ * m * pk_te.mult).sum())
        tt_rows.append({"下限": t, "券種": name, "賭け金": stake_, "払戻": ret_, "回収率": ret_ / stake_ if stake_ > 0 else np.nan})
by_type = pd.DataFrame(tt_rows)
by_type["賭け金の割合"] = by_type["賭け金"] / by_type.groupby("下限")["賭け金"].transform("sum")
display(by_type.pivot(index="券種", columns="下限", values="賭け金の割合").round(3)); by_type.to_csv(OUT / "final_by_type.csv", index=False)
# 月別（下限 0.5）
dates = wp.drop_duplicates("race_id").set_index("race_id")["date"]
s_rl = SETTLED[HEAD].copy()
s_rl["month"] = s_rl["race_id"].map(dates).str[:7]
by_month = s_rl.groupby("month").agg(races=("stake", "size"), 賭け金=("stake", "sum"), 払戻=("ret", "sum"))
by_month["回収率"] = by_month["払戻"] / by_month["賭け金"].replace(0, np.nan)
display(by_month.round(3)); by_month.to_csv(OUT / "final_by_month.csv")

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
EN = ["RL stake>=0", "RL stake>=0.25", "RL stake>=0.5", "RL stake>=1.0", "Tansho: model top1", "Tansho: favorite", "Fukusho: model top1", "Umaren: model top1",
      "EV>=tau equal", "EV>=tau top3", "EV top25% races", "EV top50% races", "1/4 Kelly", "Random 1 ticket"]
assert len(EN) == len(SETTLED), (len(EN), list(SETTLED))
fig, ax = plt.subplots(figsize=(9, 4.5))
for (k, s), lab in zip(SETTLED.items(), EN):
    c = s.assign(date=s["race_id"].map(dates)).sort_values(["date", "race_id"])["profit"].cumsum().to_numpy()
    ax.plot(c, label=lab, lw=2.2 if k == HEAD else 0.9)
ax.axhline(0, color="k", lw=0.5); ax.set_xlabel("race (chronological)"); ax.set_ylabel("cumulative profit (budget per race = 1)"); ax.legend(fontsize=6, ncol=2)
fig.tight_layout(); fig.savefig(OUT / "cumulative_profit_2025.png", dpi=130)
# 較正: 最終年の期待回収率の帯
T25 = TICK[TEST_Y]; d = T25.assign(x=T25["p_comb"] * T25["odds"]); d["band"] = pd.cut(d["x"], bins)
cal25 = d.groupby(["type", "band"], observed=True).agg(n=("mult", "size"), 期待回収率=("x", "mean"), 実際の回収率=("mult", "mean")).round(3)
cal25.to_csv(OUT / "ev_calibration_2025.csv")
display(cal25.unstack(0)["実際の回収率"].round(3))

In [ ]:
res = {"evaluated_at": datetime.now().isoformat(timespec="seconds"), "tag": TAG, "git_commit": GIT_COMMIT, "benter": {"a": A_B, "b": B_B, "fit_year": 2022},
       "best_config": BEST, "optuna_value": study.best_value, "test_races": int(pk_te.n_races), "tau": tau,
       "final": fin_df.reset_index().to_dict("records"), "vs_baselines": diff_df.reset_index().to_dict("records"),
       "notes": ["オッズは締切後の確定オッズ（購入時点の値ではない）", "三連複・三連単は対象外（オッズなし）", "複勝・ワイドは下限オッズで判断、払戻は確定値"]}
(OUT / "report.json").write_text(json.dumps(res, ensure_ascii=False, indent=1, default=str), encoding="utf-8")
if not SMOKE:
  with mlflow.start_run(run_name=f"tp-08-betting-rl-{TAG}", tags={"notebook": "tp-08", "git_commit": GIT_COMMIT, "test_year": "2025", "train": "2022-2024"}):
      mlflow.log_params({**{f"cfg_{k}": v for k, v in BEST.items()}, "benter_a": A_B, "benter_b": B_B, "K": K, "tag": TAG, "oof": OOF_FILE.name})
      for t in TARGETS:
          r0 = fin_df.loc[f"RL（下限{t}）"]
          mlflow.log_metrics({f"test2025_roi_t{t}": float(r0["roi"]), f"test2025_roi_lo_t{t}": float(r0["roi_lo"]), f"test2025_roi_hi_t{t}": float(r0["roi_hi"]),
                              f"test2025_stake_t{t}": float(r0["stake_per_race"]), f"test2025_participation_t{t}": float(r0["participation"])})
      mlflow.log_metrics({"optuna_best_valid_profit": float(study.best_value)})
      for p in OUT.glob("*"):
          mlflow.log_artifact(str(p), "betting_rl")
if not SMOKE:
    LOCK.write_text(json.dumps({"evaluated_at": res["evaluated_at"], "git_commit": GIT_COMMIT, "note": "test 2025 の馬券評価は実施済み。結果を見てやり直さない。"}, ensure_ascii=False))
print("完了:", {k: round(float(v), 4) for k, v in fin_df.loc[HEAD].items()})